In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Penanda aras litar dinamik dengan pasangan Bell yang dipotong

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Anggaran penggunaan: 22 saat pada pemproses Heron r2 (NOTA: Ini adalah anggaran sahaja. Masa jalan sebenar anda mungkin berbeza.)*
## Hasil pembelajaran
Selepas melalui tutorial ini, pengguna harus memahami:
* Cara membina Circuit dinamik dengan pengukuran litar tengah dan suap balik klasik ke depan untuk mengangkut pembelitan antara qubit yang jauh;
* Cara mengira dan mentafsir metrik ralat bagi mengukur kesetiaan pasangan Bell merentasi sebuah peranti;
* Cara mengenal pasti pasangan qubit yang paling sesuai untuk operasi berasaskan LOCC menggunakan hasil penanda aras.
## Prasyarat
Kami mencadangkan pengguna membiasakan diri dengan topik-topik berikut sebelum melalui tutorial ini:
* [Konsep pengkomputeran kuantum asas](/learning/courses/basics-of-quantum-information), termasuk keadaan Bell, pembelitan, dan gate kuantum;
* Kebiasaan dengan [Circuit dinamik](/guides/classical-feedforward-and-control-flow) ([pengukuran litar tengah](/guides/execute-dynamic-circuits#mid-circuit-measurements) dan suap balik klasik ke depan);
* Pengetahuan asas tentang [Qiskit SDK](https://docs.quantum.ibm.com/guides) dan [Qiskit Runtime](/guides/compute-services#qiskit-runtime), serta akses kepada [akaun IBM Quantum&reg;](/guides/cloud-setup).
## Latar belakang
Perkakasan kuantum biasanya terhad kepada interaksi tempatan, tapi banyak algoritma memerlukan pembelitan qubit yang jauh atau bahkan [qubit pada pemproses berasingan](#references). Litar dinamik — iaitu litar dengan pengukuran pertengahan litar dan suap balik ke depan — menyediakan cara untuk mengatasi batasan ini dengan menggunakan komunikasi klasik masa nyata untuk melaksanakan operasi kuantum bukan setempat secara berkesan. Dalam pendekatan ini, hasil pengukuran dari satu bahagian litar (atau satu QPU) boleh mencetuskan gate secara bersyarat pada bahagian lain, membolehkan kita mengangkut pembelitan merentasi jarak jauh. Ini membentuk asas skim **operasi setempat dan komunikasi klasik (LOCC)**, di mana kita menggunakan keadaan sumber yang terikat (pasangan Bell) dan berkomunikasi hasil pengukuran secara klasik untuk menghubungkan qubit yang jauh.

Salah satu kegunaan LOCC yang menjanjikan ialah untuk merealisasikan gate CNOT jarak jauh maya melalui teleportasi, seperti yang ditunjukkan dalam [tutorial pembelitan jarak jauh](/tutorials/long-range-entanglement). Daripada CNOT jarak jauh terus (yang mungkin tidak dibenarkan oleh sambungan perkakasan), kita cipta pasangan Bell dan lakukan pelaksanaan gate berasaskan teleportasi. Walau bagaimanapun, ketepatan operasi sedemikian bergantung pada ciri-ciri perkakasan. Penyahaskoheranan qubit semasa kelewatan yang diperlukan (semasa menunggu hasil pengukuran) dan kependaman komunikasi klasik boleh merendahkan keadaan yang terikat. Selain itu, ralat pada pengukuran pertengahan litar lebih sukar untuk diperbetulkan berbanding ralat pada pengukuran akhir kerana ia merambat ke seluruh litar melalui gate bersyarat.

Dalam [eksperimen rujukan](#references), para pengarang memperkenalkan penanda aras ketepatan pasangan Bell untuk mengenal pasti bahagian mana dalam peranti yang paling sesuai untuk pembelitan berasaskan LOCC. Ideanya ialah untuk menjalankan litar dinamik kecil pada setiap kumpulan empat qubit yang bersambung dalam pemproses. Litar empat-Qubit ini pertama sekali mencipta pasangan Bell pada dua qubit tengah, kemudian menggunakannya sebagai sumber untuk membelit dua qubit tepi menggunakan LOCC. Secara konkrit, qubit 1 dan 2 disediakan secara setempat dalam pasangan Bell _tidak dipotong_ (pasangan Bell yang dicipta terus dengan Hadamard dan CNOT, tanpa teleportasi), dan kemudian rutin teleportasi menggunakan pasangan Bell itu untuk membelit qubit 0 dan 3. Qubit 1 dan 2 diukur semasa pelaksanaan litar, dan berdasarkan hasil tersebut, pembetulan Pauli (X pada qubit 3 dan Z pada qubit 0) dikenakan. Qubit 0 dan 3 kemudiannya ditinggalkan dalam keadaan Bell di penghujung litar.

Untuk mengukur kualiti pasangan terikat akhir ini, kita ukur penstabilnya: khususnya, pariti dalam asas $Z$ ($Z_0Z_3$) dan dalam asas $X$ ($X_0X_3$). Untuk pasangan Bell yang sempurna, kedua-dua jangkaan ini bersamaan +1. Dalam amalan, hingar perkakasan akan mengurangkan nilai-nilai ini. Oleh itu, kita ulang litar dua kali untuk setiap pasangan qubit: satu litar mengukur qubit 0 dan 3 dalam asas $Z$, dan satu lagi mengukurnya dalam asas $X$. Daripada hasilnya, kita mendapat anggaran $\langle Z_0Z_3\rangle$ dan $\langle X_0X_3\rangle$ untuk pasangan qubit tersebut. Kita gunakan min kuasa dua ralat (MSE) penstabil ini berbanding nilai ideal (1) sebagai metrik mudah untuk ketepatan pembelitan. MSE yang lebih rendah bermakna dua qubit mencapai keadaan Bell yang lebih hampir kepada ideal (ketepatan lebih tinggi), manakala MSE yang lebih tinggi menunjukkan lebih banyak ralat. Dengan mengimbas eksperimen ini merentasi peranti, kita boleh menanda aras keupayaan pengukuran-dan-suap-balik kumpulan qubit yang berbeza dan mengenal pasti pasangan qubit terbaik untuk operasi LOCC.

Tutorial ini menunjukkan eksperimen pada peranti IBM Quantum&reg; untuk menggambarkan cara litar dinamik boleh digunakan untuk menjana dan menilai pembelitan antara qubit yang jauh. Kita akan memetakan semua rantai empat-Qubit linear pada peranti, menjalankan litar teleportasi pada setiap satunya, kemudian menggambarkan taburan nilai MSE. Prosedur hujung-ke-hujung ini menunjukkan cara memanfaatkan Qiskit Runtime dan ciri litar dinamik untuk memaklumkan pilihan yang sedar perkakasan bagi memotong litar atau mengedarkan algoritma kuantum merentasi sistem modular.
## Keperluan
Sebelum memulakan tutorial ini, pastikan anda telah memasang yang berikut:

* Qiskit SDK v2.0 atau lebih baharu, dengan sokongan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.40 atau lebih baharu (`pip install qiskit-ibm-runtime`)
* Qiskit Aer v0.17 atau lebih baharu (`pip install qiskit-aer`)
## Persediaan

In [ ]:
from qiskit import QuantumCircuit

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler import generate_preset_pass_manager

import numpy as np
import matplotlib.pyplot as plt


def create_bell_stab(initial_layouts):
    """
    Create a circuit for a 1D chain of qubits (number
    of qubits must be a multiple of 4), where a middle
    Bell pair is consumed to create a Bell at the edge.
    Takes as input a list of lists, where each element
    of the list is a 1D chain of physical qubits that is
    used as the initial_layout for the transpiled circuit.
    Returns a list of length-2 tuples, each tuple
    contains a circuit to measure the ZZ stabilizer and
    a circuit to measure the XX stabilizer of the edge
    Bell state.
    """
    bell_circuits = []
    for (
        initial_layout
    ) in initial_layouts:  # Iterate over chains of physical qubits
        assert (
            len(initial_layout) % 4 == 0
        ), f"The length of the chain must be a multiple of 4, len(inital_layout)={len(initial_layout)}"
        num_pairs = len(initial_layout) // 4

        bell_parallel = QuantumCircuit(4 * num_pairs, 4 * num_pairs)

        for pair_idx in range(num_pairs):
            (q0, q1, q2, q3) = (
                pair_idx * 4,
                pair_idx * 4 + 1,
                pair_idx * 4 + 2,
                pair_idx * 4 + 3,
            )
            (c0, c1) = pair_idx * 4, pair_idx * 4 + 3  # edge qubits
            (ca0, ca1) = pair_idx * 4 + 1, pair_idx * 4 + 2  # middle qubits

            bell_parallel.h(q0)
            bell_parallel.h(q1)
            bell_parallel.cx(q1, q2)
            bell_parallel.cx(q0, q1)
            bell_parallel.cx(q2, q3)
            bell_parallel.h(q2)

        # add barrier BEFORE measurements and add id in conditional
        bell_parallel.barrier()
        for pair_idx in range(num_pairs):
            (q0, q1, q2, q3) = (
                pair_idx * 4,
                pair_idx * 4 + 1,
                pair_idx * 4 + 2,
                pair_idx * 4 + 3,
            )
            (ca0, ca1) = pair_idx * 4 + 1, pair_idx * 4 + 2  # middle qubits

            bell_parallel.measure(q1, ca0)
            bell_parallel.measure(q2, ca1)
        # bell_parallel.barrier() #remove barrier after measurement

        for pair_idx in range(num_pairs):
            (q0, q1, q2, q3) = (
                pair_idx * 4,
                pair_idx * 4 + 1,
                pair_idx * 4 + 2,
                pair_idx * 4 + 3,
            )
            (ca0, ca1) = pair_idx * 4 + 1, pair_idx * 4 + 2  # middle qubits
            with bell_parallel.if_test((ca0, 1)):
                bell_parallel.x(q3)
            with bell_parallel.if_test((ca1, 1)):
                bell_parallel.z(q0)
                bell_parallel.id(q0)  # add id here for correct alignment

        bell_zz = bell_parallel.copy()
        bell_zz.barrier()
        bell_xx = bell_parallel.copy()
        bell_xx.barrier()
        for pair_idx in range(num_pairs):
            (q0, q1, q2, q3) = (
                pair_idx * 4,
                pair_idx * 4 + 1,
                pair_idx * 4 + 2,
                pair_idx * 4 + 3,
            )
            bell_xx.h(q0)
            bell_xx.h(q3)
        bell_xx.barrier()
        for pair_idx in range(num_pairs):
            (q0, q1, q2, q3) = (
                pair_idx * 4,
                pair_idx * 4 + 1,
                pair_idx * 4 + 2,
                pair_idx * 4 + 3,
            )
            (c0, c1) = pair_idx * 4, pair_idx * 4 + 3  # edge qubits

            bell_zz.measure(q0, c0)
            bell_zz.measure(q3, c1)

            bell_xx.measure(q0, c0)
            bell_xx.measure(q3, c1)

        bell_circuits.append(bell_zz)
        bell_circuits.append(bell_xx)

    return bell_circuits


def get_mse(result, initial_layouts):
    """
    given a result object and the initial layouts,
    returns a dict of layouts and their mse
    """
    layout_mse = {}
    for layout_idx, initial_layout in enumerate(initial_layouts):
        layout_mse[tuple(initial_layout)] = {}

        num_pairs = len(initial_layout) // 4

        counts_zz = result[2 * layout_idx].data.c.get_counts()
        total_shots = sum(counts_zz.values())

        # Get ZZ expectation value
        exp_zz_list = []
        for pair_idx in range(num_pairs):
            exp_zz = 0
            for bitstr, shots in counts_zz.items():
                bitstr = bitstr[::-1]  # reverse order to big endian
                b1, b0 = (
                    bitstr[pair_idx * 4],
                    bitstr[pair_idx * 4 + 3],
                )  # parse bitstring to get edge measurements for each 4-q chain
                z_val0 = 1 if b0 == "0" else -1
                z_val1 = 1 if b1 == "0" else -1
                exp_zz += z_val0 * z_val1 * shots
            exp_zz /= total_shots
            exp_zz_list.append(exp_zz)

        counts_xx = result[2 * layout_idx + 1].data.c.get_counts()
        total_shots = sum(counts_xx.values())

        # Get XX expectation value
        exp_xx_list = []
        for pair_idx in range(num_pairs):
            exp_xx = 0
            for bitstr, shots in counts_xx.items():
                bitstr = bitstr[::-1]  # reverse order to big endian
                b1, b0 = (
                    bitstr[pair_idx * 4],
                    bitstr[pair_idx * 4 + 3],
                )  # parse bitstring to get edge measurements for each 4-q chain
                x_val0 = 1 if b0 == "0" else -1
                x_val1 = 1 if b1 == "0" else -1
                exp_xx += x_val0 * x_val1 * shots
            exp_xx /= total_shots
            exp_xx_list.append(exp_xx)

        mse_list = [
            ((exp_zz - 1) ** 2 + (exp_xx - 1) ** 2) / 2
            for exp_zz, exp_xx in zip(exp_zz_list, exp_xx_list)
        ]

        print(f"layout {initial_layout}")
        for idx in range(num_pairs):
            layout_mse[tuple(initial_layout)][
                tuple(initial_layout[4 * idx : 4 * idx + 4])
            ] = mse_list[idx]
            print(
                f"qubits: {initial_layout[4*idx:4*idx+4]}, mse:, {round(mse_list[idx],4)}"
            )
            # print(f'exp_zz: {round(exp_zz_list[idx],4)}, exp_xx: {round(exp_xx_list[idx],4)}')
        print(" ")
    return layout_mse


def plot_mse_ecdfs(layouts_mse, combine_layouts=False):
    """
    Plot CDF of MSE data for multiple layouts.
    Optionally combine all data in a single CDF
    """

    if not combine_layouts:
        for initial_layout, layouts in layouts_mse.items():
            sorted_layouts = dict(
                sorted(layouts.items(), key=lambda item: item[1])
            )  # sort layouts by mse

            # get layouts and mses
            layout_list = list(sorted_layouts.keys())
            mse_list = np.asarray(list(sorted_layouts.values()))

            # convert to numpy
            x = np.array(mse_list)
            y = np.arange(1, len(x) + 1) / len(x)

            # Prepend (x[0], 0) to start CDF at zero
            x = np.insert(x, 0, x[0])
            y = np.insert(y, 0, 0)

            # Create the plot
            plt.plot(
                x,
                y,
                marker="x",
                linestyle="-",
                label=f"qubits: {initial_layout}",
            )

            # add qubits labels for the edge pairs
            for xi, yi, q in zip(x[1:], y[1:], layout_list):
                plt.annotate(
                    [q[0], q[3]],
                    (xi, yi),
                    textcoords="offset points",
                    xytext=(5, -10),
                    ha="left",
                    fontsize=8,
                )

    elif combine_layouts:
        all_layouts = {}
        all_initial_layout = []
        for (
            initial_layout,
            layouts,
        ) in layouts_mse.items():  # puts together all layout information
            all_layouts.update(layouts)
            all_initial_layout += initial_layout

        sorted_layouts = dict(
            sorted(all_layouts.items(), key=lambda item: item[1])
        )  # sort layouts by mse

        # get layouts and mses
        layout_list = list(sorted_layouts.keys())
        mse_list = np.asarray(list(sorted_layouts.values()))

        # convert to numpy
        x = np.array(mse_list)
        y = np.arange(1, len(x) + 1) / len(x)

        # Prepend (x[0], 0) to start CDF at zero
        x = np.insert(x, 0, x[0])
        y = np.insert(y, 0, 0)

        # Create the plot
        plt.plot(
            x,
            y,
            marker="x",
            linestyle="-",
            label=f"qubits: {sorted(list(set(all_initial_layout)))}",
        )

        # add qubit labels for the edge pairs
        for xi, yi, q in zip(x[1:], y[1:], layout_list):
            plt.annotate(
                [q[0], q[3]],
                (xi, yi),
                textcoords="offset points",
                xytext=(5, -10),
                ha="left",
                fontsize=8,
            )

    plt.xscale("log")
    plt.xlabel("Mean squared error of ⟨ZZ⟩ and ⟨XX⟩")
    plt.ylabel("Cumulative distribution function")
    plt.title("CDF for different initial layouts")
    plt.grid(alpha=0.3)
    plt.show()

## Contoh simulator berskala kecil

Sebelum menjalankan pada QPU sebenar, kita mengesahkan bahawa Circuit menghasilkan pasangan Bell dengan mengujinya pada simulator tanpa hingar menggunakan rantai empat-Qubit `[0, 1, 2, 3]`. Kita menggunakan Qiskit Runtime `Sampler` dengan `AerSimulator` sebagai mod Backend untuk melaksanakan Circuit.

### Langkah 1: Petakan input klasik kepada masalah kuantum

Langkah pertama adalah mencipta satu set Circuit kuantum untuk menilai semua calon pautan Bell-pair yang disesuaikan dengan topologi peranti. Kita mulakan dengan membina Circuit sedemikian menggunakan rantai empat-Qubit `[0, 1, 2, 3]`.

Rutin `create_bell_stab()` melakukan perkara berikut untuk setiap rantai:

* Sediakan pasangan Bell tengah: Gunakan Hadamard pada qubit 1 dan CNOT dari qubit 1 ke qubit 2. Ini melilit qubit 1 dan 2 (mewujudkan keadaan Bell $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$).
* Lilit qubit tepi: Gunakan CNOT dari qubit 0 ke qubit 1, dan CNOT dari qubit 2 ke qubit 3. Ini menghubungkan pasangan yang berasingan sebelum ini supaya qubit 0 dan 3 akan menjadi terikat selepas langkah seterusnya. Hadamard pada qubit 2 juga digunakan (ini, digabungkan dengan CNOT sebelumnya, membentuk sebahagian daripada pengukuran Bell pada qubit 1 dan 2). Pada ketika ini, qubit 0 dan 3 belum terikat, tetapi qubit 1 dan 2 terikat dengan mereka dalam keadaan empat-Qubit yang lebih besar.
* Pengukuran litar tengah dan suap balas: Qubit 1 dan 2 (qubit tengah) diukur dalam asas pengkomputeran, menghasilkan dua bit klasik. Berdasarkan keputusan pengukuran tersebut, kita menggunakan operasi bersyarat: jika pengukuran qubit 1 (panggil bit ini $m_{12}$) adalah 1, kita gunakan gate $X$ pada qubit 3; jika pengukuran qubit 2 ($m_{21}$) adalah 1, kita gunakan gate $Z$ pada qubit 0. Gate bersyarat ini (yang direalisasikan menggunakan konstruk `if_test`/`if_else` Qiskit) melaksanakan pembetulan teleportasi standard. Ia "membatalkan" balik Pauli rawak yang berlaku akibat memproyeksikan qubit 1 dan 2, memastikan qubit 0 dan 3 berakhir dalam keadaan Bell yang diketahui, tanpa mengira keputusan pengukuran. Selepas langkah ini, qubit 0 dan 3 seharusnya secara idealnya terikat dalam keadaan Bell $|\Phi^+\rangle$.
* Ukur penstabil pasangan Bell: Kita kemudian membahagi kepada dua versi Circuit. Dalam versi pertama, kita mengukur penstabil $ZZ$ pada qubit 0 dan 3. Dalam versi kedua, kita mengukur penstabil $XX$ pada qubit-Qubit ini.

Untuk setiap susun atur awal empat-Qubit, fungsi di atas mengembalikan dua Circuit (satu untuk pengukuran penstabil $ZZ$, satu lagi untuk $XX$). Circuit-circuit ini termasuk pengukuran litar tengah dan operasi bersyarat (if/else), yang merupakan arahan utama Circuit dinamik.

In [19]:
from qiskit_aer import AerSimulator

# 4-qubit chain for simulation
sim_layout = [[0, 1, 2, 3]]

aer_backend = AerSimulator()
sim_circuits = create_bell_stab(sim_layout)
sim_circuits[1].draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/tutorials/edc-cut-bell-pair-benchmarking/extracted-outputs/160e25c5-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/edc-cut-bell-pair-benchmarking/extracted-outputs/160e25c5-0.avif)

### Langkah 2: Optimumkan masalah untuk pelaksanaan perkakasan kuantum
Sebelum melaksanakan Circuit kita, kita perlu mentranspil circuit-circuit tersebut kepada operasi gate yang disokong pada Backend yang dinyatakan. Transpilasi akan memetakan Circuit abstrak kepada qubit fizikal dan set gate Backend yang dipilih. Memandangkan kita telah memilih qubit fizikal tertentu untuk setiap rantai (dengan menyediakan `initial_layout` kepada penjana Circuit), kita menggunakan `optimization_level=0` Transpiler dengan susun atur tetap tersebut. Ini memberitahu Qiskit untuk tidak menetapkan semula qubit atau melakukan sebarang pengoptimuman berat yang boleh mengubah struktur Circuit. Kita mahu mengekalkan urutan operasi (terutamanya gate bersyarat) tepat seperti yang dinyatakan.

In [ ]:
pm_sim = generate_preset_pass_manager(
    optimization_level=0, backend=aer_backend, initial_layout=sim_layout[0]
)
isa_sim_circuits = pm_sim.run(sim_circuits)
isa_sim_circuits[1].draw("mpl", fold=-1, idle_wires=False)

### Langkah 3: Laksanakan menggunakan primitif Qiskit
Kita kini boleh menjalankan eksperimen pada Backend simulator tanpa hingar.

In [14]:
# Run on noiseless simulator
sampler_sim = Sampler(mode=aer_backend)
sim_job = sampler_sim.run(isa_sim_circuits)
sim_mse = get_mse(sim_job.result(), sim_layout)

layout [0, 1, 2, 3]
qubits: [0, 1, 2, 3], mse:, 0.0
 


### Step 4: Post-process and return result in the desired classical format


The final step is to compute the mean squared error metric (MSE) for each tested qubit group and summarize the results. For each chain, we now have the measured $\langle Z_0Z_3\rangle$ and $\langle X_0X_3\rangle$. If qubits 0 and 3 were perfectly entangled in a $|\Phi^+\rangle$ Bell state, we would expect both of these to be +1. We quantify the deviation using the MSE:

$$\text{MSE} = \frac{( \langle Z_0Z_3\rangle - 1)^2 + (\langle X_0X_3\rangle - 1)^2}{2}.$$

This value is 0 for a perfect Bell pair, and increases as the entangled state gets noisier (with random outcomes giving an expectation around 0, the MSE would approach 1). The code calculates this MSE for each four-qubit group. With no noise, we see MSE = 0 in this small-scale simulator example, as expected.

## Large-scale hardware example

Here we now put all of these details together into a singular workflow at a larger scale, which is then run on real quantum hardware.

In [2]:
service = QiskitRuntimeService()
backend = service.least_busy(operational=True)

### Langkah 4: Proses selepas dan kembalikan keputusan dalam format klasik yang dikehendaki
Langkah terakhir adalah mengira metrik ralat min kuasa dua (MSE) untuk setiap kumpulan qubit yang diuji dan merumuskan keputusan. Untuk setiap rantai, kita kini mempunyai $\langle Z_0Z_3\rangle$ dan $\langle X_0X_3\rangle$ yang diukur. Jika qubit 0 dan 3 terikat sempurna dalam keadaan Bell $|\Phi^+\rangle$, kita menjangkakan kedua-duanya bernilai +1. Kita mengukur penyelewengan menggunakan MSE:

$$\text{MSE} = \frac{( \langle Z_0Z_3\rangle - 1)^2 + (\langle X_0X_3\rangle - 1)^2}{2}.$$

Nilai ini adalah 0 untuk pasangan Bell yang sempurna, dan meningkat apabila keadaan terikat menjadi lebih berderau (dengan keputusan rawak memberikan jangkaan sekitar 0, MSE akan menghampiri 1). Kod ini mengira MSE untuk setiap kumpulan empat-Qubit. Tanpa hingar, kita dapat MSE = 0 dalam contoh simulator berskala kecil ini, seperti yang dijangkakan.
## Contoh perkakasan berskala besar
Di sini kita menggabungkan semua butiran ini ke dalam satu aliran kerja tunggal pada skala yang lebih besar, yang kemudian dijalankan pada perkakasan kuantum sebenar.

In [ ]:
from itertools import chain
from collections import defaultdict


def stripes16_from_backend(backend):
    """
    Creates stripes of 16 qubits, four non-overlapping
    four-qubit chains, that cover as much of the coupling
    map as possible. Returns any unused qubits as leftovers.
    """
    # get the undirected adjacency list
    edges = backend.coupling_map.get_edges()
    graph = defaultdict(set)
    for u, v in edges:
        graph[u].add(v)
        graph[v].add(u)

    qubits = sorted(graph)  # all qubit indices that appear

    # greedy search for 4-long linear chains (blocks) ────────────
    used = set()  # qubits already placed in a block
    blocks = []  # each block is a four-qubit list

    for q in qubits:  # deterministic order for reproducibility
        if q in used:
            continue  # already consumed by earlier block

        # depth-first "straight" walk of length 3 without revisiting nodes
        def extend(path):
            if len(path) == 4:
                return path
            tip = path[-1]
            for nbr in sorted(graph[tip]):  # deterministic
                if nbr not in path and nbr not in used:
                    maybe = extend(path + [nbr])
                    if maybe:
                        return maybe
            return None

        block = extend([q])
        if block:  # found a 4-node path
            blocks.append(block)
            used.update(block)

    # bundle four four-qubit blocks into one 16-qubit
    # stripe (max number of measurement compatible with if-else)
    stripes = [
        list(chain.from_iterable(blocks[i : i + 4]))
        for i in range(0, len(blocks) // 4 * 4, 4)  # full groups of four
    ]

    leftovers = set(qubits) - set(chain.from_iterable(stripes))
    return stripes, leftovers

In [15]:
initial_layouts, leftover = stripes16_from_backend(backend)

Next, we construct the circuit for each 16-qubit stripe using the function `create_bell_stab()`. At the end of this step, we have a list of circuits covering every four-qubit chain on the device. We then transpile and run the circuits on the real backend and post-process the results.

In [17]:
# -------------------------Step 1-------------------------
circuits = create_bell_stab(initial_layouts)

# -------------------------Step 2-------------------------
isa_circuits = []
for ind, init_layout in enumerate(initial_layouts):
    pm = generate_preset_pass_manager(
        optimization_level=0, backend=backend, initial_layout=init_layout
    )
    isa_circ = pm.run(circuits[ind * 2 : ind * 2 + 2])
    isa_circuits.extend(isa_circ)
isa_circuits[1].draw("mpl", fold=-1, idle_wires=False)

# -------------------------Step 3-------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_BDC"]
job = sampler.run(isa_circuits)

# -------------------------Step 4-------------------------
layouts_mse = get_mse(job.result(), initial_layouts)

layout [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
qubits: [0, 1, 2, 3], mse:, 0.6302
qubits: [4, 5, 6, 7], mse:, 0.0949
qubits: [8, 9, 10, 11], mse:, 0.1729
qubits: [12, 13, 14, 15], mse:, 0.0473
 
layout [16, 23, 22, 21, 17, 27, 26, 25, 18, 31, 30, 29, 19, 35, 34, 33]
qubits: [16, 23, 22, 21], mse:, 0.0533
qubits: [17, 27, 26, 25], mse:, 0.2966
qubits: [18, 31, 30, 29], mse:, 0.0447
qubits: [19, 35, 34, 33], mse:, 0.0392
 
layout [36, 41, 42, 43, 37, 45, 46, 47, 38, 49, 50, 51, 39, 53, 54, 55]
qubits: [36, 41, 42, 43], mse:, 0.1577
qubits: [37, 45, 46, 47], mse:, 0.0705
qubits: [38, 49, 50, 51], mse:, 0.2914
qubits: [39, 53, 54, 55], mse:, 0.1711
 
layout [56, 63, 62, 61, 57, 67, 66, 65, 58, 71, 70, 69, 59, 75, 74, 73]
qubits: [56, 63, 62, 61], mse:, 0.1236
qubits: [57, 67, 66, 65], mse:, 0.9969
qubits: [58, 71, 70, 69], mse:, 0.0631
qubits: [59, 75, 74, 73], mse:, 0.0301
 
layout [76, 81, 82, 83, 77, 85, 86, 87, 78, 89, 90, 91, 79, 93, 94, 95]
qubits: [76, 81, 82, 83], ms

Seterusnya, kita membina Circuit untuk setiap jalur 16-Qubit menggunakan fungsi `create_bell_stab()`. Di penghujung langkah ini, kita mempunyai senarai Circuit yang merangkumi setiap rantai empat-Qubit pada peranti. Kita kemudian mentranspil dan menjalankan Circuit pada Backend sebenar dan memproses hasilnya.

In [18]:
plot_mse_ecdfs(layouts_mse, combine_layouts=True)

<Image src="../docs/images/tutorials/edc-cut-bell-pair-benchmarking/extracted-outputs/678ddac9-0.avif" alt="Output of the previous code cell" />

## Next steps
<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following materials:
- Learn how to implement [long-range entanglement with dynamic circuits](/docs/tutorials/long-range-entanglement)
- Learn how to simulate the [kicked Ising Hamiltonian with dynamic circuits](/docs/tutorials/dc-hex-ising)
</Admonition>

## References

[\[1\] Carrera Vazquez, A., Tornow, C., Ristè, D. et al. Combining quantum processors with real-time classical communication. Nature 636, 75-79 (2024)](https://www.nature.com/articles/s41586-024-08178-2).